In [2]:
# Import the necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [3]:
# Import the dataset
df = pd.read_csv('jordan_market_dataset_2026.csv')
df.head()

,Transaction_ID,Sale_Date,Shoe_Model,Colorway,Condition,Retail_Price_USD,Resale_Price_USD,Sales_Channel,Days_in_Inventory,Profit_Margin_USD
0,TRX-100001,2024-10-23,Air Jordan 11 Retro,Sail,Used,225,194.67,GOAT,13,-30.33
1,TRX-100002,2023-10-23,Air Jordan 1 High,Mocha,Deadstock (Brand New),185,230.46,StockX,10,45.46
2,TRX-100003,2023-11-13,Air Jordan 4 Retro,Concord,Deadstock (Brand New),212,270.37,kick hub 2026,33,58.37
3,TRX-100004,2024-02-01,Air Jordan 4 Retro,Mocha,Used,223,180.43,Stadium Goods,60,-42.57
4,TRX-100005,2023-05-29,Air Jordan 1 Low,Mocha,Used,137,117.43,kick hub 2026,30,-19.57


In [4]:
df.info()
df.describe()
df.isnull().sum()
df['Condition'].value_counts()
df['Shoe_Model'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Transaction_ID     5000 non-null   object 
 1   Sale_Date          5000 non-null   object 
 2   Shoe_Model         5000 non-null   object 
 3   Colorway           5000 non-null   object 
 4   Condition          5000 non-null   object 
 5   Retail_Price_USD   5000 non-null   int64  
 6   Resale_Price_USD   5000 non-null   float64
 7   Sales_Channel      5000 non-null   object 
 8   Days_in_Inventory  5000 non-null   int64  
 9   Profit_Margin_USD  5000 non-null   float64
dtypes: float64(2), int64(2), object(6)
memory usage: 390.8+ KB


Shoe_Model
Air Jordan 4 Retro     1044
Air Jordan 3 Retro     1037
Air Jordan 1 Low       1020
Air Jordan 11 Retro     973
Air Jordan 1 High       926
Name: count, dtype: int64

In [5]:
# Step 2: Define target (profitable or not) and remove leakage columns
df_model = df.copy()

# Binary target: 1 if profitable, else 0
df_model["Target_Profitable"] = (df_model["Profit_Margin_USD"] > 0).astype(int)

# Columns not available at prediction time or identifiers
drop_cols = [
    "Transaction_ID",
    "Sale_Date",
    "Resale_Price_USD",   # unknown at prediction time
    "Profit_Margin_USD"   # directly defines target, so leakage
]

X = df_model.drop(columns=drop_cols + ["Target_Profitable"])
y = df_model["Target_Profitable"]

print("X shape:", X.shape)
print("y distribution:")
print(y.value_counts(normalize=True))

X shape: (5000, 6)
y distribution:
Target_Profitable
1    0.6132
0    0.3868
Name: proportion, dtype: float64


In [6]:
# Step 3: Train-test split + preprocessing pipeline setup
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Detect column types
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

# Stratified split because target is binary
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

# Preprocessing: scale numeric, one-hot encode categorical
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ]
)

Categorical columns: ['Shoe_Model', 'Colorway', 'Condition', 'Sales_Channel']
Numeric columns: ['Retail_Price_USD', 'Days_in_Inventory']
Train shape: (4000, 6) (4000,)
Test shape: (1000, 6) (1000,)


In [7]:
# Step 4: Train baseline SVM (RBF) + evaluate
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

# Build pipeline: preprocessing + model
svm_clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42))
])

# Train
svm_clf.fit(X_train, y_train)

# Predict
y_pred = svm_clf.predict(X_test)
y_prob = svm_clf.predict_proba(X_test)[:, 1]

# Metrics
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy: {acc:.4f}")
print(f"ROC-AUC : {auc:.4f}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.9450
ROC-AUC : 0.9582

Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.86      0.92       387
           1       0.92      1.00      0.96       613

    accuracy                           0.94      1000
   macro avg       0.96      0.93      0.94      1000
weighted avg       0.95      0.94      0.94      1000

Confusion Matrix:
 [[332  55]
 [  0 613]]


In [8]:
# Step 5: Tune SVM hyperparameters
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

# Pipeline for tuning
svm_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", SVC(probability=True, random_state=42))
])

# Parameter grid
param_grid = {
    "model__kernel": ["rbf", "linear"],
    "model__C": [0.1, 1, 10, 50],
    "model__gamma": ["scale", 0.01, 0.1],   # used by rbf, ignored by linear
    "model__class_weight": [None, "balanced"]
}

# Tune by ROC-AUC
grid = GridSearchCV(
    estimator=svm_pipe,
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)
print("Best CV ROC-AUC:", round(grid.best_score_, 4))

# Evaluate best model on test set
best_model = grid.best_estimator_
y_pred_best = best_model.predict(X_test)
y_prob_best = best_model.predict_proba(X_test)[:, 1]

acc_best = accuracy_score(y_test, y_pred_best)
auc_best = roc_auc_score(y_test, y_prob_best)

print("\nTest Accuracy:", round(acc_best, 4))
print("Test ROC-AUC :", round(auc_best, 4))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_best))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_best))

Fitting 5 folds for each of 48 candidates, totalling 240 fits
Best Params: {'model__C': 10, 'model__class_weight': 'balanced', 'model__gamma': 0.1, 'model__kernel': 'rbf'}
Best CV ROC-AUC: 0.9671

Test Accuracy: 0.943
Test ROC-AUC : 0.9667

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.86      0.92       387
           1       0.92      1.00      0.96       613

    accuracy                           0.94      1000
   macro avg       0.95      0.93      0.94      1000
weighted avg       0.95      0.94      0.94      1000

Confusion Matrix:
 [[333  54]
 [  3 610]]


In [9]:
# Step 6: Save trained model + metadata for Streamlit
import os
import json
import joblib

os.makedirs("artifacts", exist_ok=True)

# Save best tuned pipeline
joblib.dump(best_model, "artifacts/svm_profitability_pipeline.joblib")

# Save metadata (features expected by app)
metadata = {
    "feature_columns": X.columns.tolist(),
    "categorical_cols": categorical_cols,
    "numeric_cols": numeric_cols,
    "target_name": "Target_Profitable",
    "positive_class_meaning": "Profitable"
}

with open("artifacts/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved files:")
print("- artifacts/svm_profitability_pipeline.joblib")
print("- artifacts/model_metadata.json")

Saved files:
- artifacts/svm_profitability_pipeline.joblib
- artifacts/model_metadata.json
